# UQ-Edge: Does Quantization Break Model Confidence?

**Systematic study of calibration in quantized small language models.**

- 8 models (135M → 3B) × 3 quant levels (FP16, INT8, NF4) × 5 benchmarks
- Phase 1: Inference + logit capture (GPU) → saves `.npz` files
- Phase 2: Metrics computation (CPU) → ECE, AUROC, Brier
- Phase 3: Self-consistency (GPU, optional)
- Phase 4: Generate paper figures

**Runtime**: ~4-6 hours total on A100 for full Phase 1.

---

## 0. Setup

In [1]:
# Check GPU
!nvidia-smi

Mon Mar 23 07:18:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create project directory on Drive
!mkdir -p /content/drive/MyDrive/uq-edge
!mkdir -p /content/drive/MyDrive/uq-edge/raw_outputs
!mkdir -p /content/drive/MyDrive/uq-edge/results
!mkdir -p /content/drive/MyDrive/uq-edge/plots
print("Drive mounted and directories created.")

Mounted at /content/drive
Drive mounted and directories created.


In [4]:
import urllib.request
import os

# Token is stored in Colab Secrets (sidebar: key icon > add "GH_TOKEN")
# This keeps it out of the notebook file so GitHub won't block your push.
from google.colab import userdata
TOKEN = ""

REPO = "AniruddhaChattopadhyay/Research-Battery-Lora"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/uq-edge"

FILES = [
    "config.py",
    "model_utils.py",
    "data_utils.py",
    "inference.py",
    "uq_methods.py",
    "metrics.py",
    "plotting.py",
    "run_all.py",
    "quick_test.py",
]

for f in FILES:
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/uq-edge/{f}"
    req = urllib.request.Request(url, headers={"Authorization": f"token {TOKEN}"})
    content = urllib.request.urlopen(req).read()
    with open(f"{PROJECT_DIR}/{f}", "wb") as out:
        out.write(content)
    print(f"  Updated {f} ({len(content):,} bytes)")

print(f"\nAll {len(FILES)} files synced to {PROJECT_DIR}")

# Verify
py_files = [f for f in os.listdir(PROJECT_DIR) if f.endswith('.py')]
print(f"Python files present: {sorted(py_files)}")

  Updated config.py (5,779 bytes)
  Updated model_utils.py (3,390 bytes)
  Updated data_utils.py (7,334 bytes)
  Updated inference.py (7,003 bytes)
  Updated uq_methods.py (4,183 bytes)
  Updated metrics.py (4,281 bytes)
  Updated plotting.py (9,165 bytes)
  Updated run_all.py (11,951 bytes)
  Updated quick_test.py (4,598 bytes)

All 9 files synced to /content/drive/MyDrive/uq-edge
Python files present: ['config.py', 'data_utils.py', 'inference.py', 'metrics.py', 'model_utils.py', 'plotting.py', 'quick_test.py', 'run_all.py', 'uq_methods.py']


In [5]:
# Install dependencies
# Colab already has torch, transformers, etc. — just ensure versions + extras
!pip install -q bitsandbytes accelerate datasets scikit-learn seaborn scipy pandas

# Verify key imports
import torch, transformers, bitsandbytes
print(f"torch={torch.__version__}, transformers={transformers.__version__}, bnb={bitsandbytes.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00:00:0100:01
torch=2.10.0+cu128, transformers=5.0.0, bnb=0.49.2
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [6]:
# Set working directory and add to Python path
import sys
import os
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

Working directory: /content/drive/MyDrive/uq-edge


In [11]:
# Quick sanity test — runs 10 samples, verifies full pipeline
!python quick_test.py

UQ-Edge Quick Test
1. Testing imports...
   torch=2.10.0+cu128, transformers=5.0.0
   numpy=2.0.2, scipy=1.16.3
   OK

2. Testing device...
   Device: cuda
   GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB
   OK

3. Testing config...
   Models: ['SmolLM2-135M']
   Quants: ['fp16']
   Benchmarks: ['triviaqa']
   OK

4. Testing data loading...
README.md: 26.7kB [00:00, 65.1MB/s]
Resolving data files: 100% 26/26 [00:00<00:00, 226248.76it/s]
rc.nocontext/train-00000-of-00001.parque(…): 100% 55.4M/55.4M [00:02<00:00, 22.4MB/s]
rc.nocontext/validation-00000-of-00001.p(…): 100% 7.34M/7.34M [00:00<00:00, 9.06MB/s]
rc.nocontext/test-00000-of-00001.parquet: 100% 1.20M/1.20M [00:00<00:00, 1.97MB/s]
Generating train split: 100% 138384/138384 [00:00<00:00, 271664.27 examples/s]
Generating validation split: 100% 17944/17944 [00:00<00:00, 270507.86 examples/s]
Generating test split: 100% 17210/17210 [00:00<00:00, 779507.70 examples/s]
  Loaded 10 samples from triviaqa
   Sample question: Answer the foll

## 1. Phase 1 — Inference (GPU Required)

Each cell runs **one model at all quant levels and all benchmarks**. This means 3 quant × 5 bench = 15 runs per model.

**Resume-safe**: If Colab disconnects, just re-run the cell. It skips already-completed `.npz` files.

Estimated time per model: **20-40 min** (depends on model size).

In [12]:
# Model 1: SmolLM2-135M (~20 min)
!python run_all.py --phase1 --model SmolLM2-135M


MODEL: SmolLM2-135M @ fp16
  Loading HuggingFaceTB/SmolLM2-135M-Instruct (fp16)...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 272/272 [00:00<00:00, 1095.95it/s, Materializing param=model.norm.weight]                              
  Memory: 257 MB

  Benchmark: triviaqa
Resolving data files: 100% 26/26 [00:00<00:00, 168289.98it/s]
  Loaded 1000 samples from triviaqa
    [1/1000]
    [25/1000]
    [50/1000]
    [75/1000]
    [100/1000]
    [125/1000]
    [150/1000]
    [175/1000]
    [200/1000]
    [225/1000]
    [250/1000]
    [275/1000]
    [300/1000]
    [325/1000]
    [350/1000]
    [375/1000]
    [400/1000]
    [425/1000]
    [450/1000]
    [475/1000]
    [500/1000]
    [525/1000]
    [550/1000]
    [575/1000]
    [600/1000]
    [625/1000]
    [650/1000]
    [675/1000]
    [700/1000]
    [725/1000]
    [750/1000]
    [775/1000]
    [800/1000]
    [825/1000]
    [850/1000]
    [875/1000]
    [900/1000]
    [925/1000]
    [950/1000]
    [975/1000]
    [10

In [13]:
# Model 2: SmolLM2-1.7B (~25 min)
!python run_all.py --phase1 --model SmolLM2-1.7B


MODEL: SmolLM2-1.7B @ fp16
  Loading HuggingFaceTB/SmolLM2-1.7B-Instruct (fp16)...
config.json: 100% 908/908 [00:00<00:00, 5.84MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.42G/3.42G [00:09<00:00, 379MB/s]  
Loading weights: 100% 218/218 [00:01<00:00, 209.91it/s, Materializing param=model.norm.weight]                              
generation_config.json: 100% 132/132 [00:00<00:00, 665kB/s]
tokenizer_config.json: 3.76kB [00:00, 12.7MB/s]
tokenizer.json: 2.10MB [00:00, 140MB/s]
special_tokens_map.json: 100% 655/655 [00:00<00:00, 3.84MB/s]
  Memory: 3264 MB

  Benchmark: triviaqa
Resolving data files: 100% 26/26 [00:00<00:00, 212992.00it/s]
  Loaded 1000 samples from triviaqa
    [1/1000]
    [25/1000]
    [50/1000]
    [75/1000]
    [100/1000]
    [125/1000]
    [150/1000]
    [175/1000]
    [200/1000]
    [225/1000]
    [250/1000]
    [275/1000]
    [300/1000]
    [325/1000]
    [350/1000]
    [375/1000]
    [400/1000]
    [425/1000]
    [450/1000]
 

In [7]:
!python run_all.py --phase2 --summary

  SmolLM2-135M @ fp16 — triviaqa
  SmolLM2-135M @ fp16 — mmlu
  SmolLM2-135M @ fp16 — gsm8k
  SmolLM2-135M @ fp16 — truthfulqa
  SmolLM2-135M @ fp16 — csqa
  SmolLM2-135M @ int8 — triviaqa
  SmolLM2-135M @ int8 — mmlu
  SmolLM2-135M @ int8 — gsm8k
  SmolLM2-135M @ int8 — truthfulqa
  SmolLM2-135M @ int8 — csqa
  SmolLM2-135M @ nf4 — triviaqa
  SmolLM2-135M @ nf4 — mmlu
  SmolLM2-135M @ nf4 — gsm8k
  SmolLM2-135M @ nf4 — truthfulqa
  SmolLM2-135M @ nf4 — csqa
  SmolLM2-1.7B @ fp16 — triviaqa
  SmolLM2-1.7B @ fp16 — mmlu
  SmolLM2-1.7B @ fp16 — gsm8k
  SmolLM2-1.7B @ fp16 — truthfulqa
  SmolLM2-1.7B @ fp16 — csqa
  SmolLM2-1.7B @ int8 — triviaqa
  SmolLM2-1.7B @ int8 — mmlu
  SmolLM2-1.7B @ int8 — gsm8k
  SmolLM2-1.7B @ int8 — truthfulqa
  SmolLM2-1.7B @ int8 — csqa
  SmolLM2-1.7B @ nf4 — triviaqa
  SmolLM2-1.7B @ nf4 — mmlu
  SmolLM2-1.7B @ nf4 — gsm8k
  SmolLM2-1.7B @ nf4 — truthfulqa
  SmolLM2-1.7B @ nf4 — csqa
  Qwen2.5-0.5B @ fp16 — triviaqa
  Qwen2.5-0.5B @ fp16 — mmlu
  Qwen2.5-0.

In [8]:
# Model 3: Qwen2.5-0.5B (~20 min)
!python run_all.py --phase1 --model Qwen2.5-0.5B


MODEL: Qwen2.5-0.5B @ fp16
  Loading Qwen/Qwen2.5-0.5B-Instruct (fp16)...
config.json: 100% 659/659 [00:00<00:00, 3.64MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 988M/988M [00:02<00:00, 389MB/s]  
Loading weights: 100% 290/290 [00:00<00:00, 674.11it/s, Materializing param=model.norm.weight]                              
generation_config.json: 100% 242/242 [00:00<00:00, 1.68MB/s]
tokenizer_config.json: 7.30kB [00:00, 25.6MB/s]
vocab.json: 2.78MB [00:00, 48.1MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 7.03MB [00:00, 150MB/s]
  Memory: 942 MB
  SKIP triviaqa — already exists: raw_outputs/Qwen2.5-0.5B_fp16_triviaqa.npz
  SKIP mmlu — already exists: raw_outputs/Qwen2.5-0.5B_fp16_mmlu.npz
  SKIP gsm8k — already exists: raw_outputs/Qwen2.5-0.5B_fp16_gsm8k.npz
  SKIP truthfulqa — already exists: raw_outputs/Qwen2.5-0.5B_fp16_truthfulqa.npz
  SKIP csqa — already exists: raw_outputs/Qwen2.5-0.5B_fp16_csqa.npz

MODEL: Qwen2.5-0.5B @ int8
  Loadi

: 

In [ ]:
# Model 4: Qwen2.5-1.5B (~25 min)
!python run_all.py --phase1 --model Qwen2.5-1.5B

In [ ]:
# Model 5: Qwen2.5-3B (~30 min)
!python run_all.py --phase1 --model Qwen2.5-3B

In [ ]:
# Model 6: TinyLlama-1.1B (~25 min)
!python run_all.py --phase1 --model TinyLlama-1.1B

In [ ]:
# Model 7: Llama-3.2-1B (~25 min)
# Note: may need HF token for gated model. Uncomment below if needed:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
!python run_all.py --phase1 --model Llama-3.2-1B

In [ ]:
# Model 8: Llama-3.2-3B (~40 min)
!python run_all.py --phase1 --model Llama-3.2-3B

In [ ]:
# Check Phase 1 progress — how many .npz files do we have?
import os
npz_files = [f for f in os.listdir("raw_outputs") if f.endswith(".npz")]
print(f"Completed: {len(npz_files)} / 120 expected (8 models × 3 quants × 5 benchmarks)")
print("\nFiles:")
for f in sorted(npz_files):
    size = os.path.getsize(f"raw_outputs/{f}") / 1024
    print(f"  {f} ({size:.0f} KB)")

## 2. Phase 2 — Compute Metrics (CPU Only)

This is fast (~1 min). Reads `.npz` files, computes ECE/AUROC/Brier for each UQ method, saves JSON.

In [ ]:
!python run_all.py --phase2

## 3. Results Summary

In [ ]:
!python run_all.py --summary

## 4. Generate Paper Figures

In [ ]:
!python run_all.py --plots

In [ ]:
# Display key figures inline
from IPython.display import Image, display
import os

plot_dir = "plots"
key_plots = [
    "ece_heatmap_msp.png",
    "ece_delta_msp.png",
    "reliability_msp.png",
    "auroc_comparison.png",
    "ece_heatmap_temp_scaled_msp.png",
]

for fname in key_plots:
    path = os.path.join(plot_dir, fname)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"  {fname}")
        print(f"{'='*60}")
        display(Image(filename=path, width=800))

## 5. Phase 3 — Self-Consistency (Optional, GPU Required)

Generates k=5 sampled responses per question to measure agreement-based confidence.
Only needed if we want to compare sampling-based UQ against single-pass methods.

**Runtime**: ~2-3 hours additional. Run only after reviewing Phase 2 results.

In [ ]:
# Uncomment to run Phase 3
# !python run_all.py --phase3

# Then recompute metrics to include self-consistency scores
# !python run_all.py --phase2 --plots --summary

## 6. Download Results

In [ ]:
# Zip results for download (raw_outputs are already on Drive)
!cd /content/drive/MyDrive/uq-edge && zip -r /content/uq_edge_results.zip results/ plots/

# Download the zip
from google.colab import files
files.download('/content/uq_edge_results.zip')
print("Download started. Check your browser downloads.")

## Quick Reference

| What | Command |
|------|---------|
| Run everything for one model | `!python run_all.py --phase1 --model SmolLM2-135M` |
| Recompute metrics | `!python run_all.py --phase2` |
| Regenerate plots | `!python run_all.py --plots` |
| Print summary table | `!python run_all.py --summary` |
| Force re-run (ignore existing) | Add `--force` to any command |
| Run single quant level | `!python run_all.py --phase1 --model SmolLM2-135M --quant nf4` |